# Information Retrieval. Lab 3: Evaluation

## Introduction

So far we have learnt how to implement a number of retrieval models such as the Boolean Retrieval Model, Vector Space Model, and BM25, and how to build a simple search interface over a document collection.

However, building a retrieval system is only part of the task — we also need principled ways to measure how well a system performs.

In this lab, we focus on the evaluation of information retrieval systems. Using a small annotated subset of documents, we will introduce and implement several standard IR evaluation metrics, including **precision@k**, **recall@k**, **F1 score**, and **normalized discounted cumulative gain (nDCG)**.

In this lab, evaluation and interactive search are treated as two separate components. Evaluation is carried out on a small annotated subset to illustrate how the metrics work, while the search interface uses the full collection to demonstrate practical retrieval.


In [ ]:
# We fix the NumPy version here to ensure this lab runs properly.
# You may see warnings about conflicts with some Colab pre-installed packages.
# These warnings can be safely ignored and will not affect the exercises in this lab.
!pip uninstall -y numpy
!pip install -q numpy==1.26.4

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np
import urllib.request
from bs4 import BeautifulSoup
import re
#If imports fail after reinstalling NumPy, this is likely due to binary incompatibilities caused by changing NumPy versions within the same kernel. Please restart the session and try again.

First lets retrieve and process some documents to use.

In [ ]:
def get_boat_poems():
  poems = []
  poem_titles = []
  # get poems in HTML
  url = 'https://discoverpoetry.com/poems/poems-about-ships/'
  contents = urllib.request.urlopen(url).read()
  soup = BeautifulSoup(contents)

  pattern_html = re.compile('<.*?>')

  for poem_html in soup.find_all('article', {'class': 'poem-listing'}):
    # This still contains HTML tags.
    # We need to remove everything between each pair of "<" and ">":
    poem = re.sub(pattern_html, '', str(poem_html))

    # Each document contains "\xa0Full Text" which is not meaningful for our
    # search. It could potentially negatively affect the search ranking since
    # the term "text" is in the english vocabulary. Let's remove it:
    poem = poem.replace('\xa0Full Text', '')

    # In one of the poems, some of the words are emphasised with underscores.
    # "bells_" and "bells" would be classed as different terms by
    # CountVectorizer. This may be useful for searching some dataset,
    # for example, ones that contain programming code. However, in the case of
    # poems we most likely want them removed:
    poem = poem.replace('_', '')

    # Remove for cosmetic reasons:
    poem = poem.replace('►▼', '')

    poems.append(poem)

  # Similarly, we can retrieve the titles in a separate list:
  for title_html in soup.find_all('h3', {'class': 'cat-poem-title'}):
    poem_titles.append(re.sub(pattern_html, '', str(title_html)))

  # select first 12 poems
  poems = poems[:12]
  poem_titles = poem_titles[:12]

  return poems, poem_titles

In [ ]:
# index here = webpage index - 1
corpus, titles = get_boat_poems()
for doc_id, title in enumerate(titles):
  print(f'{doc_id}: {title}')

In [ ]:
# Let's print out an example to check if it make sense and matches the website:
print(corpus[2])

## Data Preprocessing

The poems contain lots of the same word but with different plurality and tenses. Extra preprocessing using NLTK's `WordNetLemmantizer` will fix that.

Some other tokens which contain punctuation or single letter words can also slip through, so additional filters are put in place using the python package `re`.



In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

# custom preprocessing with a lemmatization and punctuation / short word removal

class LemmaTokenizer:

    def __init__(self):
        self.wnl = WordNetLemmatizer()

    def __call__(self, doc):

        regex_num_punctuation = r'(\d+)|([^\w\s])'
        regex_short_words = r'(\b\w{1,2}\b)'

        return [
            self.wnl.lemmatize(t.lower())
            for t in word_tokenize(doc)
            if not re.search(regex_num_punctuation, t)
            and not re.search(regex_short_words, t)
        ]
# vectorize the data

vectorizer = CountVectorizer(tokenizer=LemmaTokenizer(), stop_words='english')
documents_vectorized = vectorizer.fit_transform(corpus)
vocabulary = vectorizer.get_feature_names_out()

In [ ]:
print ('We have a {} document corpus with a {} term vocabulary'.format(*documents_vectorized.shape))

In [ ]:
# always useful to check the vocab at this stage, to see check for any unwanted tokens.

vocabulary

In [ ]:
# This is what it looks like
df = pd.DataFrame(documents_vectorized.toarray(), columns=vocabulary)
doc_ids = df.index.values
df

In [ ]:
def BM25_IDF_df(df):
  """
  This definition calculates BM25-IDF weights before hand as done last week
  """

  tfs = df.div(df.sum(axis=1), axis=0)
  dfs = (df > 0).sum(axis=0)
  N = df.shape[0]
  idfs = np.log(N / dfs)

  k_1 = 1.2
  b = 0.8
  dls = df.sum(axis=1)
  avgdl = np.mean(dls)

  numerator = np.array((k_1 + 1) * tfs)
  denominator = np.array(k_1 *((1 - b) + b * (dls / avgdl))).reshape(-1,1) + np.array(tfs)

  BM25_tf = numerator / denominator

  idfs = np.array(idfs)

  BM25_score = BM25_tf * idfs
  return pd.DataFrame(BM25_score, columns=vocabulary)


In [ ]:
bm25_df = BM25_IDF_df(df) # a dataframe with BM25-idf weights
bm25_df

In order to evaluate a search engine over this data we need two things:
1. Queries
2. Relevance judgements (is the document actually relevant for a given query?)

Below we will create a relevance judgement list that will determine how relevant each document is for a given query. This is equivalent to giving a label or 'ground truth' to a sample in machine learning.

Some have been left blank, so please fill them in with your own opinion of whether you think the document is relevant for the query.

Remember, all relevance judgements are subjective, and there will be some variance between the judgement of different people. You may find some relevance judgements below that you disagree with! feel free to update them if so.

If you are having trouble reading or understanding the poems, then you can use ChatGPT to summarise them.

In [ ]:
print(corpus[11])

In [ ]:
# QUERIES dictionary with {query_id: query}
queries = dict(enumerate(['love heart joy',
                          'wind rain storm']))

# RELEVANCE JUDGEMENTS list with [(query_id, document_id, judgement), ...]

# Relevance jugement score scale:
#   2 - very relevant
#   1 - somewhat relevant
#   0 - not relevant

rel_judgements = [(0, 0, 0),
         (0, 1, 2),
         (0, 2, 0),
         (0, 3, 2),
         (0, 4, 1),
         (0, 5, ...), # ... enter your relevance judgement here
         (0, 6, 1),
         (0, 7, 0),
         (0, 8, 0),
         (0, 9, 1),
         (0, 10, ...), # ... enter your relevance judgement here
         (0, 11, ...), # ... enter your relevance judgement here

         (1, 0, 1),
         (1, 1, 0),
         (1, 2, 0),
         (1, 3, 1),
         (1, 4, ...), # ... enter your relevance judgement here
         (1, 5, 1),
         (1, 6, 2),
         (1, 7, 0),
         (1, 8, 0),
         (1, 9, 2),
         (1, 10, ...), # ... enter your relevance judgement here
         (1, 11, ...)] # ... enter your relevance judgement here

Lets retrieve some results from our BM25 model for these two queries

In [ ]:
def retrieve_ranking(query, bm25_df):
  q_terms = query.split(' ')
  q_terms_only = bm25_df[q_terms]
  score_q_d = q_terms_only.sum(axis=1)
  return sorted(zip(bm25_df.index.values, score_q_d.values),
                key = lambda tup:tup[1],
                reverse=True)


# Let's look at the first few scores for our query and document combinations:
for count, query in enumerate(queries.values()):
  print(f'Query {count}: {query}')
  print('')
  ranking = retrieve_ranking(query, bm25_df)
  for idx, score in ranking:
    print(f' Doc {idx}:  {titles[idx]}, Score {score:}')
  print('')

Take a look at some of the top ranked results for each query. Do these make sense?

In [ ]:
# Next, we will define functions for calculating precision and F1-score.
# For the sake of calculating these metrics, let's reduce the jugement scores to binary judgements.
rel_judgements_binary = [(query_id, doc_id, int(judgement > 0)) for query_id, doc_id, judgement in rel_judgements]
rel_judgements_binary

## Precision

Precision indicates the accuracy of the retrieved results. How many of the retrieved results are relevant?

It can be calculated using this formula.

$ \text{Precision} = \frac{\text{Number of relevant documents retrieved}}{\text{Total number of documents retrieved}} = \frac{\text{True Positives}}{\text{True Positives + Fase Positives}} $

True positives are relevant documents that have been retrieved, while False positives are irrelevant documents that have been retrieved.

A high precision indicates that a large proportion of the retrieved documents are relevant to the query, while a low precision suggests that many irrelevant documents are being retrieved along with the relevant ones.

However, there is a problem with this metric. In many Information Retrieval algorithms, many (sometimes all) of the documents can be retrieved in a ranked list. Does this mean that the model thinks all of these are relevant? The answer is no, only the top scoring results should be considered. To fix this problem, it is very common to use Preicision @ K, where only the top K results are considered 'retrieved' documents. Typical values for K are 5, 10, 15.



In [ ]:
# TASK 1: Finish writing a function for calculate precision @ k, where k is
# the number of top ranking documents retrieved.
# HINT: Think about precision in terms of true positives (TP) and false positives. (FP)
# HINT: Remember set theory from Lab 1 and 2? TP and FP can each be considered an intersection between two sets of documents.


def precision_at_k(query_id, k=5):
  """This function considers the k top ranking documents."""
  doc_ranking = retrieve_ranking(queries[query_id], bm25_df)

  # take only the document id, rather than score
  retrieved = [doc[0] for doc in doc_ranking[:k]]

  # filter the relevant judgements to the ones which are relevant to the given query
  # .... your code goes here

  # retrieve the ids of documents that have positive relevance judgements (i.e. relevant documents)
  # .... your code goes here

  # retrieve the ids of documents that have 0 relevance judgements (i.e. non relevant documents)
  # .... your code goes here

  # number of true positives (HINT: we are trying to find the intersection between two sets)
  TP = # .... your code goes here

  # number of false positives (HINT: we are trying to find the intersection between two sets)
  FP = # .... your code goes here.

  # calculate precision
  precision = # ... your code goes here

  return TP, FP, precision


# Let's see what we get when we consider the top 5 ranking documents:
def print_precision_for_all_queries(k=5):
  for query_id, query in queries.items():
    TP, FP, precision = precision_at_k(query_id, k=k)
    print(f'retrieved query "{query}" with precision @ {k}: {precision} (TP: {TP}, FP: {FP})')

print_precision_for_all_queries(k=5)

## Recall

Recall is a metric that measures the ability of the system to retrieve all relevant documents from a collection. i.e What proportion of the relevant documents have retrieved.

It can be calculated using this formula.

$ \text{Recall} = \frac{\text{Number of relevant documents retrieved}}{\text{Total number of relevant documents}} = \frac{\text{True Positives}}{\text{True Positives + False Negatives}} $

We have discussed True Positives and False Positives, but what are False Negatives..? These are relevant documents that have not been retrieved.

Recall is particularly important in information retrieval tasks where it is crucial not to miss any relevant information, such as in legal document retrieval, medical information retrieval, or any other domain where completeness of results is critical.

Like in precision, we also use recall @ K to only consider the top K results


In [ ]:
# TASK 2: Finish writing a function for calculate recall @ k, where k is
# the number of top ranking documents retrieved.
# HINT: Think about recall in terms of true positives (TP) and false positives. (FN)
# HINT: Remember set theory from Lab 1 and 2? TP and FN can each be considered an intersection between two sets of documents.


def recall_at_k(query_id, k=5):
  """This function considers the k top ranking documents."""
  doc_ranking = retrieve_ranking(queries[query_id], bm25_df)

  # take only the document id, rather than score
  retrieved = [doc[0] for doc in doc_ranking[:k]]

  # filter the relevant judgements to return only the ones which are relevant to the given query
  # .... your code goes here

  # retrieve the ids of documents that have positive relevance judgements (i.e. relevant documents)
  # .... your code goes here

  # number of true positives (HINT: we are trying to find the intersection between two sets)
  TP = # .... your code goes here

  # number of false negatives (HINT: we are trying to find the difference between two sets)
  FN = # .... your code goes here.

  # calculate recall
  recall = # .... your code goes here.

  return TP, FN, recall


# Let's see what we get when we consider the top 5 ranking documents:
def print_recall_for_all_queries(k=5):
  for query_id, query in queries.items():
    TP, FN, recall = recall_at_k(query_id, k=k)
    print(f'retrieved query "{query}" with recall @ {k}: {recall} (TP: {TP}, FN: {FN})')

print_recall_for_all_queries(k=5)

## F1 Score

F1 score is a single metric that quantifies the performance of a retrieval system by combining precision and recall.

It can be calculated using this formula.

$ \text{F1} =  2 * \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}} $



In [ ]:
# TASK 3: Calculate F1-Score.


def f1_score_at_k(query_id, k=5):
  """This function considers the k top ranking documents."""
  doc_ranking = retrieve_ranking(queries[query_id], bm25_df)

  # call you functions recall_at_k and precision_at_k
  # your code goes here ....


  f1 = # your code goes here ...

  return f1


# To retrieve and calculate F1-Score for each query lets loop over them:
def print_f1_score_for_all_queries(k=5):
  for query_id, query in queries.items():
    f1_score = f1_score_at_k(query_id, k=k)
    print(f'retrieved query "{query}" with F1-score @ {k}: {f1_score}')

print_f1_score_for_all_queries(k=5)

## Normalized Discounted Cumulative Gain (nDCG)

So far we have only used metrics that consider binary relevance judgements (0 or 1) to quantify the performance of our retrieval model. However, when determining whether each document was relevant or not, we actually used a categorical label: 0 (not relevant), 1 (somewhat relevant) and 2 (very relevant).

Normalised Discounted Cumulative Gain (nDCG) is a metric that takes into account it accounts for a document's relevance category and also its ranking in the search results.

To recap on the method to calculated nDCG, have a look at IR lecture slides from week 5 on Evaluation, slides 49-58.

In [ ]:
# TASK 4:

# Have a look at the documentation of sklearn NDCG implementation:
# https://scikit-learn.org/stable/modules/generated/sklearn.metrics.ndcg_score.html
from sklearn.metrics import ndcg_score

# The input parameters you need are "y_true", "y_score", and "k".
# "y_true" comes from qrels (you might want to use the original qrels with the
# more nuanced judgement scores).
# "y_score" comes from document rankings.
# "y_true" and "y_score" should be Numpy arrays.
# For (n_samples, n_labels) in the documentation, here n_labels = 1.

## Interactive Search Interface

So far, we have focused on how to evaluate retrieval systems using standard IR metrics.

In this section, we shift our attention back to the system itself. We will set up ElasticSearch.

We are aware that many students may not have prior experience with web development or front-end tools. Therefore, this section demonstrates simple notebook-friendly UI components that are easy to customise and suitable for your assessment submission.


In [ ]:
# The following bash scripts download the ElasticSearch library and install it
# on the Google Colab instance.
# Credit for the bash scripts and server run start goes to:
# https://gist.github.com/korakot/15fe4f18d0e0f53d7b834ef797880500

# You need to run these only once when you work on your search engine notebook.

# NOTE: If you are working on a large dataset (20k+ docs) you should do this
# locally i.e. in a jupyter notebook. This way you only need to install ES once
# and index your data once.

In [ ]:
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512
!tar -xzf elasticsearch-oss-7.9.2-linux-x86_64.tar.gz
!sudo chown -R daemon:daemon elasticsearch-7.9.2/
!shasum -a 512 -c elasticsearch-oss-7.9.2-linux-x86_64.tar.gz.sha512

In [ ]:
# https://stackoverflow.com/questions/68762774/elasticsearchunsupportedproducterror-the-client-noticed-that-the-server-is-no#answer-68918449
!pip install elasticsearch==7.9.1 -q

In [ ]:
# check elasticsearch version in environment
!pip freeze | grep elasticsearch

In [ ]:
import time
# let's import ES
from elasticsearch import Elasticsearch

In [ ]:
%%bash --bg
sudo -H -u daemon elasticsearch-7.9.2/bin/elasticsearch

In [ ]:
%%bash
ps -ef | grep elasticsearch

In [ ]:
# start ES server
time.sleep(30) # give the server 30 seconds to start ..
!curl -X GET "http://localhost:9200"

In [ ]:
# start and test es
es = Elasticsearch("http://localhost:9200")
if es.ping():
  print('ES instance working')
else:
  print('ES instance not working')

In [ ]:
# Server information
es.info()

## Data Indexing

Next, we want to index some data. Let's use the poems from the previous lab, but this time let's add document IDs and poem lengths. Let's change the mapping a little bit to help with some filtering tasks we will do later.

In [ ]:
def get_boat_poems_full():
  poems = []
  lengths = []
  titles = []
  authors = []
  # get poems in HTML
  url = 'https://discoverpoetry.com/poems/poems-about-ships/'
  contents = urllib.request.urlopen(url).read()
  soup = BeautifulSoup(contents)

  pattern_html = re.compile('<.*?>')

  for poem_html in soup.find_all('article', {'class': 'poem-listing'}):
    # This still contains HTML tags.
    # We need to remove everything between each pair of "<" and ">":
    poem = re.sub(pattern_html, '', str(poem_html))

    # Each document contains "\xa0Full Text" which is not meaningful for our
    # search. It could potentially negatively affect the search ranking since
    # the term "text" is in the english vocabulary. Let's remove it:
    poem = poem.replace('\xa0Full Text', '')

    # In one of the poems, some of the words are emphasised with underscores.
    # "bells_" and "bells" would be classed as different terms by
    # CountVectorizer. This may be useful for searching some dataset,
    # for example, ones that contain programming code. However, in the case of
    # poems we most likely want them removed:
    poem = poem.replace('_', '')

    # Remove for cosmetic reasons:
    poem = poem.replace('►▼', '')

    poems.append(poem)

    # this includes title and author, stop words, and characters like "—"
    lengths.append(len(poem.split()))

  # Similarly, we can retrieve the titles in a separate list:
  for title_html in soup.find_all('h3', {'class': 'cat-poem-title'}):
    titles.append(re.sub(pattern_html, '', str(title_html)))

  for author in soup.find_all('div', {'class': 'intro'}):
    authors.append(re.sub(pattern_html, '', str(author)).replace('by ', ''))

  doc_ids = range(len(poems))

  corpus = []
  for doc_id, poem, length, title, author in zip(doc_ids, poems, lengths, titles, authors):
    corpus.append((doc_id, poem, length, title, author))

  return corpus

In [ ]:
corpus_full = get_boat_poems_full()

In [ ]:
# es.indices.delete(index_name)

In [ ]:
# Mappings are used to define what kind of structure your data has.
# Here an explicit mapping is used:
# https://www.elastic.co/guide/en/elasticsearch/reference/current/explicit-mapping.html

# The mapping is used when creating the index through the request body:

request_body = {
    'settings': {
        'number_of_shards': 1,
        'number_of_replicas': 1
    },
    'mappings': {
          'properties': {
              'doc_id': {'type': 'integer'},
              'body': {'type': 'text'},
              'length': {'type': 'integer'},
              'title': {'type': 'text'},
              'author': {'type': 'keyword'}
          }
    }
}

# Notice that we set "author" field type to "keyword". We will need
# to perform aggregations (e.g. finding the minimum value accross all documents)
# later.

index_name = 'poems-about-ships'
try:
  es.indices.get(index_name)
  print('index {} already exists'.format(index_name))
except:
  print('creating index {}'.format(index_name))
  es.indices.create(index_name, body=request_body)

In [ ]:
# Now what we want to do is to put some data into the index, i.e. index it:
for doc_id, poem, length, title, author in corpus_full:
  doc_body = {
      'doc_id': doc_id,
      'body': poem,
      'length': length,
      'title': title,
      'author': author
  }
  es.index(index_name, doc_body)

In [ ]:
def index_info(index_name):
  count, deleted, shards, =  es.cat.indices(index=index_name,
                                            h=['docs.count', 'docs.deleted', 'pri'])[:-1].split(' ')
  print(
      """
      #### INDEX INFO #####
      index_name = {}
      doc_count = {}
      shard_count = {}
      deleted_doc_count = {}
      """.format(index_name, count, shards, deleted)
  )


index_info(index_name)

In [ ]:
def search(index_name, query_body):
  results = es.search(index=index_name, body=query_body, explain=False)
  plain_results = []  # We fields we are most interested in
  for hit in results['hits']['hits']:
    plain_results.append((hit['_source']['doc_id'],
                          hit['_source']['title'],
                          hit['_source']['author'],
                          hit['_source']['length'],
                          hit['_score']))
  return results, plain_results

In [ ]:
# Let's define a function for printing out the results in an easy-to-read
# format:


def print_plain_results(plain_results):
  print('Doc ID', ' ' * 54, 'Title', ' ' * 23, 'Author', ' ' * 1, 'Length', ' ' * 2, 'Score')
  print('-' * 116)
  for doc_id, title, author, length, score in plain_results:
    print(f'{doc_id: 6} {title: >60} {author: >30} {length: >8}    {score:.2f}')

In [ ]:
query_body = {
    'query':{
        'term': {
            'body':  'ship'
        }
    }
}
results, plain_results = search(index_name, query_body)
print_plain_results(plain_results)

## Simple UI with Colab Forms

Let's consider a situation where a user wants to find a poem to memorize. They have length requirements - the user doesn't want to spend a lot of time and effort to memorize an extremely long poem, but a very short poem would not impress their peers. Let's create a user interface to help them find the right
poem. Sliders seem like a good option.


In [ ]:
# Let's start with finding the lengths of the shortest and the longest poems so
# that we know what the slider range should be.
# To do that we can use ElasticSearch "aggregations". See the documentation:
# https://www.elastic.co/guide/en/elasticsearch/reference/7.9/search-aggregations-metrics-min-aggregation.html
# https://www.elastic.co/guide/en/elasticsearch/reference/7.9/search-aggregations-metrics-max-aggregation.html

query_body = {
  "aggs": {
    "min_length": { "min": { "field": "length" }},
    "max_length": { "max": { "field": "length" }}
  }
}
result = es.search(index=index_name, body=query_body)

min_length = result['aggregations']['min_length']['value']
max_length = result['aggregations']['max_length']['value']

if min_length is None or max_length is None:
    raise RuntimeError("Elasticsearch aggregations not ready yet — rerun cell.")

print(f"Minimum length: {int(min_length)}")
print(f"Maximum length: {int(max_length)}")


In the example below, a simple boolean model is used to do retrieval.

The code can be hidden by clicking the three vertical dot symbol on the top right of the cell or the arrow on the top left, resulting in a simple UI.

This can be customised using any model and filter.

In [ ]:
#@title Search Engine: Retrieve Poems and Filter by Length

Query = 'rain, storm'  #@param {type:"string"}
Min_length = 287 #@param {type: "slider", min: 10, max: 1000, step: 1}
Max_length = 352 #@param {type: "slider", min: 10, max: 1000, step: 1}

query_body = {
  "query": {
    "bool": {
      "should": [
        {
          "terms": {
            "body": Query.split(' ')
          }
        }
      ],
      "must": [
        {
          "range": {
            "length": {
              "gte": Min_length,  # greater-than or equal-to
              "lte": Max_length  # less-than or equal-to
            }
          }
        }
      ]
    }
  }
}

results, plain_results = search(index_name, query_body)
print_plain_results(plain_results)

## UI using Jupyter Widgets

What if a user wants to find all poems by a specific author? Let's help them by creating a dropdown menu.

In [ ]:
import ipywidgets as widgets

In [ ]:
# First, let's obtain all unique authors. We can do that by using our original
# data or the index.

# Get unique authors from our original data:
unique_authors = set([doc[4] for doc in corpus_full])

# Alternatively, get unique authors from the index:
num_docs = es.count(index=index_name)['count']
query_body = {
  "size": 0,
  "aggs": {
    "authors": {
        "terms": { "field": "author", "size": num_docs }
    }
  }
}
results = es.search(index=index_name, body=query_body)
unique_authors = [bucket['key'] for bucket in results['aggregations']['authors']['buckets']]

# Next, let's use widgets this time so that we could pass the list of unique
# authors as variable to a dropdown menu.
author_picker = widgets.Dropdown(options=unique_authors, value='Emily Dickinson')

In [ ]:
display(author_picker)

query_body = {
  "query": {
    "bool": {
      "must": [
        {
          "match": {
            "author": author_picker.value
          }
        }
      ]
    }
  }
}

results, plain_results = search(index_name, query_body)
print_plain_results(plain_results)

## Resources

###ElasticSearch API

* [Creating an Index](https://www.elastic.co/guide/en/elasticsearch/reference/current/indices-create-index.html)
* [Using IR models](https://www.elastic.co/guide/en/elasticsearch/reference/current/index-modules-similarity.html)
* [Mappings](https://www.elastic.co/guide/en/elasticsearch/reference/current/mapping.html)
* [Shards](https://opster.com/guides/elasticsearch/glossary/elasticsearch-shards/)

### More UI Examples

* [Colab Forms](https://colab.research.google.com/notebooks/forms.ipynb#scrollTo=eFN7-fUKs-Bu)
* [Jupyter Widgets](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html#intrangeslider)
